# Desafio Final — Store Sales: Time Series Forecasting

**Competição Kaggle:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)

Neste desafio, vamos prever as **vendas diárias** de milhares de combinações loja × família de produto da **Corporación Favorita**, uma grande rede de supermercados do Equador.

Este notebook implementa um **baseline completo**, do carregamento dos dados até a geração do arquivo `submission.csv`. O objetivo não é atingir o melhor score possível, mas criar uma **base sólida e interpretável** que possa ser evoluída.

Leia o arquivo `plan.md` nesta mesma pasta para entender a estratégia e os próximos passos de melhoria.

## 1. Setup e Imports

Vamos importar todas as bibliotecas necessárias. Usaremos:
- **pandas / numpy**: manipulação de dados
- **matplotlib / seaborn**: visualizações
- **lightgbm**: o modelo de gradient boosting que usaremos no baseline
- **sklearn**: métricas de avaliação

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error

import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Bibliotecas carregadas com sucesso!")
print(f"XGBoost version: {xgb.__version__}")


## 2. Carregamento dos Dados

Os dados estão em arquivos `.zip`. Vamos carregá-los diretamente sem precisar extraí-los manualmente.

Temos **7 arquivos** diferentes para explorar. Além do `train.csv` e `test.csv`, os arquivos auxiliares (`stores`, `oil`, `holidays_events`, `transactions`) contêm informações externas que podem enriquecer nosso modelo.

In [ ]:
DATA_DIR = "/kaggle/input/competitions/store-sales-time-series-forecasting/"

train        = pd.read_csv(DATA_DIR + "train.csv",             parse_dates=["date"])
test         = pd.read_csv(DATA_DIR + "test.csv",              parse_dates=["date"])
stores       = pd.read_csv(DATA_DIR + "stores.csv")
oil          = pd.read_csv(DATA_DIR + "oil.csv",               parse_dates=["date"])
holidays     = pd.read_csv(DATA_DIR + "holidays_events.csv",   parse_dates=["date"])
transactions = pd.read_csv(DATA_DIR + "transactions.csv",      parse_dates=["date"])
sample_sub   = pd.read_csv(DATA_DIR + "sample_submission.csv")

print(f"train:        {train.shape}")
print(f"test:         {test.shape}")
print(f"stores:       {stores.shape}")
print(f"oil:          {oil.shape}")
print(f"holidays:     {holidays.shape}")
print(f"transactions: {transactions.shape}")


In [ ]:
train["family"] = train["family"].astype("category")
test["family"] = test["family"].astype("category")

## 3. Análise Exploratória (EDA)

Antes de modelar, precisamos **entender os dados profundamente**. Uma boa EDA revela padrões, anomalias e oportunidades de feature engineering. Vamos explorar cada aspecto importante do dataset.

### 3.1 Estrutura e Tipos de Dados

In [ ]:
print("=== train.csv ===")
print(train.info())
print("\nPrimeiras linhas:")
train.head()

In [ ]:
display("Store", stores.head())
display("Oil", oil.head())
display("Holidays", holidays.head())
display("Transactions", transactions.head())

In [ ]:
print(f"Período de treino: {train['date'].min().date()} → {train['date'].max().date()}")
print(f"Período de teste:  {test['date'].min().date()} → {test['date'].max().date()}")
print(f"\nLojas únicas:           {train['store_nbr'].nunique()}")
print(f"Famílias únicas:        {train['family'].nunique()}")
print(f"Combinações lojaxfamília: {train.groupby(['store_nbr','family']).ngroups}")
print(f"\nFração de vendas = 0:   {(train['sales'] == 0).mean():.1%}")

### 3.2 Distribuição das Vendas

A distribuição das vendas é extremamente assimétrica (muitos zeros e alguns valores muito altos). Por isso, a métrica RMSLE e o uso de `log1p` como transformação do target são escolhas naturais — elas "comprimem" a escala e tratam vendas altas e baixas de forma mais equilibrada.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribuição original (apenas valores > 0 para melhor visualização)
non_zero = train.loc[train["sales"] > 0, "sales"]
axes[0].hist(non_zero.clip(upper=non_zero.quantile(0.99)), bins=60, color="steelblue", edgecolor="white")
axes[0].set_title("Distribuição das Vendas (excl. zeros, clipado no p99)")
axes[0].set_xlabel("Vendas")
axes[0].set_ylabel("Frequência")

# Distribuição com log1p
axes[1].hist(np.log1p(train["sales"]), bins=60, color="darkorange", edgecolor="white")
axes[1].set_title("Distribuição de log1p(Vendas)")
axes[1].set_xlabel("log1p(Vendas)")
axes[1].set_ylabel("Frequência")

plt.tight_layout()
plt.show()

print(f"Vendas — estatísticas básicas:")
print(train["sales"].describe().round(2))

### 3.3 Evolução das Vendas ao Longo do Tempo

Vamos agregar as vendas totais por dia para ver a tendência geral. Isso revela sazonalidade anual, crescimento ao longo dos anos, e anomalias (como o terremoto de 2016).

In [ ]:
daily_sales = train.groupby("date")["sales"].sum().reset_index()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(daily_sales["date"], daily_sales["sales"], color="steelblue", linewidth=0.8, alpha=0.8)

# Marcar o terremoto de Abril/2016
quake_date = pd.Timestamp("2016-04-16")
ax.axvline(quake_date, color="red", linestyle="--", linewidth=1.5, label="Terremoto (16/04/2016)")

ax.set_title("Vendas Totais por Dia (todas as lojas e famílias)")
ax.set_xlabel("Data")
ax.set_ylabel("Vendas Totais")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

### 3.4 Padrão Semanal

O dia da semana é uma das features mais poderosas em varejo. Supermercados costumam ter picos no fim de semana. Vamos confirmar isso nos dados.

In [ ]:
day_names = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]

weekly = (
    train.copy()
    .assign(day_of_week=lambda df: df["date"].dt.dayofweek)
    .groupby("day_of_week")["sales"]
    .mean()
    .reset_index()
)
weekly["day_name"] = weekly["day_of_week"].map(dict(enumerate(day_names)))

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#e74c3c" if d >= 5 else "steelblue" for d in weekly["day_of_week"]]
ax.bar(weekly["day_name"], weekly["sales"], color=colors, edgecolor="white")
ax.set_title("Venda Média por Dia da Semana")
ax.set_ylabel("Venda Média")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

### 3.5 Famílias e Lojas com Maior Volume de Vendas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top famílias
top_families = train.groupby("family")["sales"].sum().sort_values(ascending=True).tail(15)
axes[0].barh(top_families.index, top_families.values, color="steelblue")
axes[0].set_title("Top 15 Famílias por Venda Total")
axes[0].set_xlabel("Vendas Totais")

# Top lojas
top_stores = (
    train.groupby("store_nbr")["sales"].sum().sort_values(ascending=True).tail(15)
)
axes[1].barh([f"Loja {s}" for s in top_stores.index], top_stores.values, color="darkorange")
axes[1].set_title("Top 15 Lojas por Venda Total")
axes[1].set_xlabel("Vendas Totais")

plt.tight_layout()
plt.show()

### 3.6 Preço do Petróleo

O Equador é fortemente dependente da receita do petróleo. Quedas no preço afetam a renda do governo e, indiretamente, o poder de compra da população. Vamos ver a evolução do preço e seu alinhamento com as vendas.

In [ ]:
oil_filled = oil.set_index("date")["dcoilwtico"].resample("D").interpolate("linear").reset_index()
oil_filled.columns = ["date", "oil_price"]

fig, ax1 = plt.subplots(figsize=(16, 5))
ax2 = ax1.twinx()

ax1.plot(daily_sales["date"], daily_sales["sales"], color="steelblue", alpha=0.5, linewidth=0.8, label="Vendas totais")
ax2.plot(oil_filled["date"], oil_filled["oil_price"], color="darkred", linewidth=1.2, alpha=0.8, label="Preço do petróleo")

ax1.set_ylabel("Vendas Totais", color="steelblue")
ax2.set_ylabel("Preço do Petróleo (USD)", color="darkred")
ax1.set_title("Vendas Totais vs Preço do Petróleo")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()
plt.show()

print(f"Valores nulos no preço do petróleo (original): {oil['dcoilwtico'].isna().sum()}")
print(f"Após interpolação linear: {oil_filled['oil_price'].isna().sum()}")

## 4. Feature Engineering

Aqui transformamos os dados brutos em features que o modelo pode usar. O princípio é: **extrair do contexto tudo que pode explicar por que as vendas sobem ou caem em um determinado dia**.

### Estratégia de Features

1. **Features de data**: captura padrões cíclicos (dia da semana, mês, etc.)
2. **Flags de negócio**: feriados, dias de pagamento — eventos que impactam o comportamento do consumidor
3. **Dados externos**: preço do petróleo, tipo e cluster da loja
4. **Encoding de categorias**: famílias de produto e lojas como inteiros (LightGBM lida bem com isso)

In [ ]:
# Feriados nacionais (excluindo transferidos, pois o dia real não é feriado)
national_holidays = holidays[
    (holidays["locale"] == "National") & (~holidays["transferred"])
]["date"].unique()

def build_features(df):
    df = df.copy()

    # --- Features de data ---
    df["day_of_week"]  = df["date"].dt.dayofweek        # 0=Seg, 6=Dom
    df["day_of_month"] = df["date"].dt.day
    df["month"]        = df["date"].dt.month
    df["year"]         = df["date"].dt.year
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["is_weekend"]   = (df["day_of_week"] >= 5).astype(int)

    # --- Dias de pagamento (salários públicos no dia 15 e último do mês) ---
    last_days = df["date"].dt.days_in_month
    df["is_payday"] = ((df["day_of_month"] == 15) | (df["day_of_month"] == last_days)).astype(int)

    # --- Feriado nacional ---
    df["is_holiday"] = df["date"].isin(national_holidays).astype(int)

    # --- Preço do petróleo (interpolado) ---
    df = df.merge(oil_filled, on="date", how="left")
    df["oil_price"] = df["oil_price"].ffill().bfill()

    # --- Metadados das lojas ---
    df = df.merge(stores, on="store_nbr", how="left")

    # --- Encoding de categorias ---
    df["family_enc"] = pd.Categorical(df["family"]).codes
    df["type_enc"]   = pd.Categorical(df["type"]).codes
    df["city_enc"]   = pd.Categorical(df["city"]).codes
    df["state_enc"]  = pd.Categorical(df["state"]).codes

    return df

train_fe = build_features(train)
test_fe  = build_features(test)

print("Features geradas. Colunas do dataset de treino:")
print(train_fe.columns.tolist())
display(train_fe.head())

## 5. Modelo Baseline — XGBoost (GPU)

### Por que XGBoost em vez de LightGBM?

Ambos são algoritmos de gradient boosting excelentes, mas diferem na arquitetura de GPU:

| | LightGBM GPU | XGBoost GPU (`device="cuda"`) |
|---|---|---|
| O que roda na GPU | Só construção de histogramas | Histogramas + split finding + tree building |
| Utilização típica de GPU | 10–20% | 60–90% |

### Estratégia de crescimento das árvores: leaf-wise vs level-wise

Por padrão, XGBoost cresce as árvores **level-wise**: completa todos os nós de profundidade N antes de avançar para N+1 — produz árvores simétricas mas desperdiça splits em regiões pouco informativas.

LightGBM usa **leaf-wise**: a cada passo, expande apenas a folha com o maior ganho de loss — concentra a capacidade do modelo onde ela é mais útil.

Para equiparar o comportamento, usamos `grow_policy="lossguide"` + `max_leaves=128` no XGBoost. Sem isso, o score fica ~0.78 RMSE; com isso, deve chegar próximo ao 0.45 do LightGBM.

### Transformação do Target

Continuamos usando **`log1p(sales)`** como target e **`expm1()`** para reverter as previsões.

In [ ]:
FEATURES = [
    "store_nbr",
    "family_enc",
    "onpromotion",
    "day_of_week",
    "day_of_month",
    "month",
    "year",
    "week_of_year",
    "is_weekend",
    "is_payday",
    "is_holiday",
    "oil_price",
    "type_enc",
    "city_enc",
    "state_enc",
    "cluster",
]

TARGET = "sales"

X_train = train_fe[FEATURES]
y_train = np.log1p(train_fe[TARGET])   # transformação log1p

X_test = test_fe[FEATURES]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test  shape: {X_test.shape}")

### 5.1 Validação com Holdout Temporal

Para estimar a performance real do modelo, vamos separar os **últimos 15 dias do treino** como conjunto de validação. Isso simula exatamente a situação do teste: prever 15 dias à frente.

> **Importante**: Em séries temporais, NUNCA use validação cruzada aleatória. O modelo treinado em dados do futuro para prever o passado é inválido — os dados do futuro não estariam disponíveis na vida real.

In [ ]:
cutoff_date = train_fe["date"].max() - pd.Timedelta(days=15)

mask_val = train_fe["date"] > cutoff_date
mask_tr  = ~mask_val

X_tr  = X_train[mask_tr]
y_tr  = y_train[mask_tr]
X_val = X_train[mask_val]
y_val = y_train[mask_val]

print(f"Treino:    {X_tr.shape[0]:,} linhas  ({train_fe.loc[mask_tr, 'date'].min().date()} → {train_fe.loc[mask_tr, 'date'].max().date()})")
print(f"Validação: {X_val.shape[0]:,} linhas  ({train_fe.loc[mask_val, 'date'].min().date()} → {train_fe.loc[mask_val, 'date'].max().date()})")

In [ ]:
params = {
    "device":           "cuda",
    "tree_method":      "hist",
    "grow_policy":      "lossguide",   # leaf-wise, como o LightGBM
    "max_leaves":       128,           # equivalente a num_leaves=127 do LightGBM
    "max_depth":        0,             # sem limite de profundidade no modo lossguide
    "objective":        "reg:squarederror",
    "eval_metric":      "rmse",
    "learning_rate":    0.05,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 20,
    "seed":             42,
}

dtrain = xgb.DMatrix(X_tr,  label=y_tr)
dval   = xgb.DMatrix(X_val, label=y_val)

print("Treinando modelo (GPU)...")

evals_result = {}
model = xgb.train(
    params,
    dtrain,
    num_boost_round=5000,
    evals=[(dtrain, "train"), (dval, "valid")],
    early_stopping_rounds=50,
    evals_result=evals_result,
    verbose_eval=100,
)

print(f"\nMelhor iteração: {model.best_iteration}")

In [ ]:
val_preds_log = model.predict(xgb.DMatrix(X_val))
val_preds     = np.expm1(val_preds_log).clip(0)
y_val_orig    = np.expm1(y_val)

rmsle_val = np.sqrt(mean_squared_log_error(y_val_orig, val_preds))
print(f"RMSLE na validação (holdout): {rmsle_val:.4f}")
print()
print("Quanto menor o RMSLE, melhor. Um score < 0.5 já é razoável para um baseline.")

### 5.2 Importância das Features

Vamos verificar quais features o modelo considera mais relevantes. Isso nos dá insights valiosos: features importantes merecem mais atenção no refinamento; features irrelevantes podem ser removidas.

In [ ]:
importance = model.get_score(importance_type="gain")
feat_imp = pd.Series(importance).reindex(FEATURES, fill_value=0).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Importância das Features (Gain)")
ax.set_xlabel("Importância Total (Gain)")
plt.tight_layout()
plt.show()

## 6. Treino Final e Geração da Submissão

Agora que validamos o modelo, vamos **treinar com todos os dados de treino disponíveis** (sem holdout). Quanto mais dados o modelo vê, melhor ele aprende — e não precisamos mais de validação porque já estimamos nossa performance.

Em seguida, geramos as previsões para o conjunto de teste e salvamos o arquivo de submissão.

In [ ]:
dtrain_full = xgb.DMatrix(X_train, label=y_train)

print(f"Treinando modelo final com {X_train.shape[0]:,} exemplos e {model.best_iteration} iterações...")

model_final = xgb.train(
    params,
    dtrain_full,
    num_boost_round=model.best_iteration,
    verbose_eval=200,
)

print("Treino final concluído!")

In [ ]:
test_preds_log = model_final.predict(xgb.DMatrix(X_test))
test_preds     = np.expm1(test_preds_log).clip(0)

submission = sample_sub.copy()
submission["sales"] = test_preds

submission.to_csv("/kaggle/working/submission.csv", index=False)

print(f"submission.csv salvo com {len(submission):,} linhas.")
print(f"\nEstatísticas das previsões:")
print(submission["sales"].describe().round(2))
submission.head(10)


In [ ]:
# Sanidade: verificar que não há negativos e que o formato está correto
assert (submission["sales"] >= 0).all(), "Há valores negativos nas previsões!"
assert list(submission.columns) == ["id", "sales"], "Colunas incorretas!"
assert len(submission) == len(sample_sub), "Número de linhas não bate com sample_submission!"

print("✓ Verificação de sanidade passou.")
print(f"✓ {len(submission):,} previsões geradas.")
print(f"✓ Nenhum valor negativo.")
print(f"✓ Formato correto: {list(submission.columns)}")
print()
print("O arquivo '/kaggle/working/submission.csv' está pronto para submissão!")
print("Vá em Output → Submit to Competition para enviar.")


## 7. Próximos Passos

Este baseline nos dá um ponto de partida funcional. Agora o desafio real começa: **como melhorar o score?**

Abaixo estão os caminhos mais promissores, em ordem de impacto esperado:

---

### Alto Impacto — Lag Features

As vendas do passado próximo são os melhores preditores das vendas futuras. Adicionar **lag features** é geralmente a melhoria mais significativa em problemas de séries temporais:

```python
# Exemplos de lag features para adicionar ao build_features()
# Atenção: use apenas lags >= 16 para não vazar dados do futuro
train_sorted = train.sort_values(["store_nbr", "family", "date"])
train_sorted["lag_16"] = train_sorted.groupby(["store_nbr", "family"])["sales"].shift(16)
train_sorted["lag_21"] = train_sorted.groupby(["store_nbr", "family"])["sales"].shift(21)
train_sorted["lag_28"] = train_sorted.groupby(["store_nbr", "family"])["sales"].shift(28)
train_sorted["rolling_mean_28"] = (
    train_sorted.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(16).rolling(28).mean())
)
```

> **Por que lag >= 16?** O período de teste são os 15 dias seguintes. Se o modelo for usado para prever o dia 16 (o primeiro do teste), o valor do dia 15 (lag=1) ainda está disponível. Mas para prever todos os 15 dias de teste, precisamos de lags que não dependam do período de teste.

---

### Médio Impacto — Feriados Enriquecidos

Adicionar a **proximidade de feriados** (dias antes/depois) captura o efeito de antecipação de compras:

```python
# Quantos dias faltam para o próximo feriado nacional?
all_dates = pd.date_range("2013-01-01", "2017-08-31")
holiday_set = set(pd.to_datetime(national_holidays))
days_to_next = {d: min([(h - d).days for h in holiday_set if h >= d], default=999) for d in all_dates}
```

---

### Médio Impacto — Validação Mais Robusta (TimeSeriesSplit)

Em vez de um único holdout, use múltiplas janelas de validação para ter uma estimativa mais confiável da performance:

```python
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5, gap=0, test_size=15 * 1782)
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    # treinar e avaliar cada fold
    pass
```

---

### Explorações Avançadas

| Técnica | Descrição |
|---|---|
| **Prophet** | Modelo do Meta para séries temporais com múltiplas sazonalidades |
| **Modelos por família** | Treinar um modelo separado para cada família de produto |
| **Tratamento do terremoto** | Excluir ou ponderar menos o período Abr–Jun 2016 |
| **Ensemble** | Combinar previsões de LightGBM + outro modelo |
| **Neural Networks** | N-BEATS, TFT — state-of-the-art em séries temporais |

---

### Como Submeter no Kaggle

1. Faça upload deste notebook para o Kaggle (ou execute diretamente lá)
2. Certifique-se de que o notebook gera o arquivo `submission.csv` como output
3. Vá em **Output → Submit to Competition**

Boa sorte!